In [1]:
import jax
import netket as nk
from copy import deepcopy

import numpy as np
import jax.numpy as jnp

# from neuralimportancesampling._src.driver.ngd_antoine.grad_sample.models import PCMolecule
from neuralimportancesampling._src.driver.ngd_antoine.grad_sample.models import PCMolecule 


from netket.experimental.operator._particle_number_conserving_fermionic._kernels import get_conn_padded_pnc, get_conn_padded_pnc_spin, unpack_spin_sectors
from netket.experimental.operator._particle_number_conserving_fermionic._matrix_elements import hamming_distance

/home/filippo/Desktop/Project8/venvs/nis/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
mol, mo_coeff, mf = PCMolecule.molecule(cid=947)#62714
molecule = PCMolecule(mol=mol, mo_coeff=mo_coeff, conserve_spin=True)

H = molecule.hamiltonian.to_jax_operator()
hi = molecule.hilbert_space

Hartree-Fock energy: -107.49896754458382
E(CCSD) = -107.6560799993205  E_corr = -0.1571124547366467
CCSD energy: -107.65607999932047


/home/filippo/Desktop/Project8/venvs/nis/lib/python3.12/site-packages/jax/_src/ops/scatter.py:108: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=int64 to dtype=bool with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(


In [ ]:
all_states = hi.all_states()
print("all_states_size =", all_states.shape)

_operator_data = H._operator_data
print("_operator_data keys =", _operator_data.keys())

all_states_size = (14400, 20)
_operator_data keys = dict_keys(['diag', 'offdiag', 'mixed_diag', 'mixed_offdiag'])


: 

In [4]:
x = all_states[np.random.randint(0, len(all_states))]
x

Array([1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1], dtype=int8)

**1. Offdiag elements**

In [5]:
def filter_keys(pytree, filter_func:callable):
    result = {}
    for outer_key, inner_dict in pytree.items():
        result[outer_key] = {
            (k,s): v for (k,s), v in inner_dict.items() 
            if filter_func(k,s)
        }
    return result

_operator_data_filtered = filter_keys(_operator_data, lambda k,s: k == 4)
_operator_data_filtered['diag'] = _operator_data_filtered['mixed_diag'] = _operator_data_filtered['mixed_offdiag'] = {}

In [6]:
import jax.ops
from netket.experimental.operator._particle_number_conserving_fermionic._kernels import get_conn_padded_pnc_spin
from netket.experimental.operator._particle_number_conserving_fermionic._matrix_elements import _get_mel_offdiag, _get_mel_mixed_offdiag

# x = jnp.array([1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0])

xp, mels = get_conn_padded_pnc_spin(_operator_data_filtered, x, hi.n_fermions_per_spin)
xp, inverse_indices = jnp.unique(xp, axis=0, return_inverse=True)
mels = jax.ops.segment_sum(mels, inverse_indices, num_segments=len(xp))

print("xp shape =", xp.shape)
print("mels shape =", mels.shape)

xp shape = (34, 20)
mels shape = (34,)


In [7]:
_xp, _ = get_conn_padded_pnc_spin(_operator_data, x, hi.n_fermions_per_spin)
_xp, _ = jnp.unique(_xp, axis=0, return_inverse=True)

n_fermions_per_spin = hi.n_fermions_per_spin
n_spin_subsectors = len(n_fermions_per_spin)

xs = unpack_spin_sectors(x, n_spin_subsectors)  # ((n,), (n,))
ys = unpack_spin_sectors(_xp, n_spin_subsectors)  # ((batch, n), (batch, n))

mels_offdiag = jnp.zeros(_xp.shape[0], dtype=mels.dtype)
for (k, sectors), v in _operator_data_filtered['offdiag'].items():
    assert k == 4, "This loop should only process k == 4 terms"
    for i in sectors:
        j = 1 - i  # other sector than the one selected. assume 2 sectors: 0 and 1
        is_allowed = jnp.all(
            xs[j] == ys[j], axis=-1
        )  # 2-body transitions within the same spin sector i are only allowed if the other sector j remains unchanged
        mels_offdiag += (
            _get_mel_offdiag(n_fermions_per_spin[0], xs[i], ys[i], *v) * is_allowed
        )

In [8]:
jnp.round(jnp.unique(mels[mels != 0], axis=0) - jnp.unique(mels_offdiag[mels_offdiag != 0], axis=0), decimals=15)

Array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],      dtype=float64)

In [9]:
jnp.max(jnp.unique(mels[mels != 0], axis=0) - jnp.unique(mels_offdiag[mels_offdiag != 0], axis=0))

Array(2.77555756e-17, dtype=float64)

**2. Mixed offdiag elements**

In [10]:
def filter_keys(pytree, filter_func:callable):
    result = {}
    for outer_key, inner_dict in pytree.items():
        result[outer_key] = {
            (k,s): v for (k,s), v in inner_dict.items() 
            if filter_func(k,s)
        }
    return result

_operator_data_filtered = filter_keys(_operator_data, lambda k,s: k == 4)
_operator_data_filtered['diag'] = _operator_data_filtered['mixed_diag'] = _operator_data_filtered['offdiag'] = {}

In [11]:
import jax.ops
from netket.experimental.operator._particle_number_conserving_fermionic._kernels import get_conn_padded_pnc_spin
from netket.experimental.operator._particle_number_conserving_fermionic._matrix_elements import _get_mel_offdiag, _get_mel_mixed_offdiag

# x = jnp.array([1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0])

xp, mels = get_conn_padded_pnc_spin(_operator_data_filtered, x, hi.n_fermions_per_spin)
xp, inverse_indices = jnp.unique(xp, axis=0, return_inverse=True)
mels = jax.ops.segment_sum(mels, inverse_indices, num_segments=len(xp))

print("xp shape =", xp.shape)
print("mels shape =", mels.shape)

xp shape = (92, 20)
mels shape = (92,)


In [12]:
_xp, _ = get_conn_padded_pnc_spin(_operator_data, x, hi.n_fermions_per_spin)
_xp = jnp.unique(_xp, axis=0)

n_fermions_per_spin = hi.n_fermions_per_spin
n_spin_subsectors = len(n_fermions_per_spin)

xs = unpack_spin_sectors(x, n_spin_subsectors)  # ((n,), (n,))
ys = unpack_spin_sectors(_xp, n_spin_subsectors)  # ((batch, n), (batch, n))

mels_mixed_offdiag = jnp.zeros(_xp.shape[0], dtype=mels.dtype)
for (k, sectors), v in _operator_data_filtered['mixed_offdiag'].items():
    assert k == 4, "This loop should only process k == 4 terms"

    mels_mixed_offdiag += _get_mel_mixed_offdiag(n_fermions_per_spin[0], xs[0], xs[1], ys[0], ys[1], *v)

In [13]:
jnp.round(jnp.unique(mels[mels != 0], axis=0) - jnp.unique(mels_mixed_offdiag[mels_mixed_offdiag != 0], axis=0), decimals=15)

Array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float64)

In [14]:
jnp.max(jnp.unique(mels[mels != 0], axis=0) - jnp.unique(mels_mixed_offdiag[mels_mixed_offdiag != 0], axis=0))

Array(0., dtype=float64)

**3. Full truncated get_conn_pad**

In [15]:
from netket.experimental.operator._particle_number_conserving_fermionic._kernels_truncated import get_conn_padded_pnc_spin_truncated

# x = jnp.array([1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0])
xp, mels = get_conn_padded_pnc_spin(_operator_data, x, hi.n_fermions_per_spin)
xp, inverse_indices = jnp.unique(xp, axis=0, return_inverse=True)
mels = jax.ops.segment_sum(mels, inverse_indices, num_segments=len(xp))

In [16]:
_xp, _ = get_conn_padded_pnc_spin(_operator_data, x, hi.n_fermions_per_spin)
_xp = jnp.unique(_xp, axis=0)

xp_truncated, mels_truncated = get_conn_padded_pnc_spin_truncated(
    _operator_data,
    x,
    _xp,
    hi.n_fermions_per_spin,
)
xp_truncated, inverse_indices = jnp.unique(xp_truncated, axis=0, return_inverse=True)
mels_truncated = jax.ops.segment_sum(mels_truncated, inverse_indices, num_segments=len(xp_truncated))


In [17]:
jnp.round(jnp.abs(mels - mels_truncated), decimals=15)

Array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0.], dtype=float64)

In [18]:
jnp.max(jnp.abs(mels - mels_truncated))

Array(8.32667268e-17, dtype=float64)

**4 Partial truncated get_conn_pad**

In [19]:
all_connected, _ = get_conn_padded_pnc_spin(_operator_data, x, hi.n_fermions_per_spin)
y_reduced = all_connected[0:100]

# all diagonal and 1-body nondiagonal terms
_operator_data_reduced = deepcopy(_operator_data)
_operator_data_reduced['offdiag'] = {
    key: val for key, val in _operator_data['offdiag'].items() if key[0] < 4
}
_operator_data_reduced['mixed_offdiag'] = {
    key: val for key, val in _operator_data['mixed_offdiag'].items() if key[0] < 4
}

xp_reduced, mels_reduced = get_conn_padded_pnc_spin(_operator_data_reduced, x, hi.n_fermions_per_spin)

# only k=4 two-body offdiagonal terms (the ones on which we truncate)
_operator_data_twobody_offdiag = deepcopy(_operator_data)
_operator_data_twobody_offdiag['diag'] = {}
_operator_data_twobody_offdiag['mixed_diag'] = {}
_operator_data_twobody_offdiag['offdiag'] = {
    key: val for key, val in _operator_data['offdiag'].items() if key[0] == 4
}
_operator_data_twobody_offdiag['mixed_offdiag'] = {
    key: val for key, val in _operator_data['mixed_offdiag'].items() if key[0] == 4
}

# actual truncation
xp_twobody_offdiag, mels_twobody_offdiag = get_conn_padded_pnc_spin(_operator_data_twobody_offdiag, x, hi.n_fermions_per_spin)
mask = (xp_twobody_offdiag[:, None] == y_reduced[None, :]).all(axis=-1).any(axis=-1)
xp_twobody_offdiag_truncated = xp_twobody_offdiag[mask]
mels_twobody_offdiag_truncated = mels_twobody_offdiag[mask]

xp = jnp.concatenate([xp_reduced, xp_twobody_offdiag_truncated], axis=-2)
mels = jnp.concatenate([mels_reduced, mels_twobody_offdiag_truncated], axis=-1)
xp, inverse_indices = jnp.unique(xp, axis=0, return_inverse=True)
mels = jax.ops.segment_sum(mels, inverse_indices, num_segments=len(xp))

In [20]:
y_reduced = jnp.unique(y_reduced, axis=0)

xp_truncated, mels_truncated = get_conn_padded_pnc_spin_truncated(
    _operator_data,
    x,
    y_reduced,
    hi.n_fermions_per_spin,
)
xp_truncated, inverse_indices = jnp.unique(xp_truncated, axis=0, return_inverse=True)
mels_truncated = jax.ops.segment_sum(mels_truncated, inverse_indices, num_segments=len(xp_truncated))


In [21]:
jnp.round(jnp.abs(mels - mels_truncated), decimals=15)

Array([0., 0., 0., 0., 0., 0., 0.], dtype=float64)

In [ ]:
jnp.max(jnp.abs(mels - mels_truncated))

Array(8.32667268e-17, dtype=float64)

: 